In [ ]:
import pandas as pd

df = pd.read_csv('data/online_retail_II.csv')

print(df.head())
print(df.info())
print(df.describe())

# Sprawdź braki danych
print(df.isnull().sum())

In [ ]:
import pandas as pd

# --- 1. Load Data (Adjust path if needed) ---
print("Loading data...")
df = pd.read_csv('data/online_retail_II.csv')

# --- 2. Apply Logic (Inline for testing) ---
# Copy to avoid side effects
df_test = df.copy()

# A. Basic Cleaning
df_test = df_test.dropna(subset=['Customer ID'])
df_test['Customer ID'] = df_test['Customer ID'].astype(int).astype(str)
df_test['InvoiceDate'] = pd.to_datetime(df_test['InvoiceDate'])
df_test['TotalAmount'] = df_test['Quantity'] * df_test['Price']

# B. Splitting Sales vs Returns
sales_df = df_test[df_test['Quantity'] > 0]
returns_raw = df_test[df_test['Quantity'] < 0]
# C. Creating Metrics for Returns
# We take absolute value to sum up "how much money was returned"
returns_raw['AbsReturnAmount'] = returns_raw['TotalAmount'].abs()

returns_metrics = returns_raw.groupby('Customer ID').agg(
    ReturnsCount=('Invoice', 'count'),
    TotalReturnedValue=('AbsReturnAmount', 'sum')
).reset_index()

# --- 3. Verification ---

print(f"\nOriginal rows: {len(df)}")
print(f"Sales rows (Clean): {len(sales_df)}")
print(f"Customers with returns: {len(returns_metrics)}")

print("\n--- SAMPLE: Sales Data (What we will use for RFM) ---")
print(sales_df[['Customer ID', 'Invoice', 'Quantity', 'TotalAmount']].head())

print("\n--- SAMPLE: Return Metrics (The new table) ---")
print(returns_metrics.head())

# Optional: Find a specific customer who has returns to check if math is right
sample_return_customer = returns_metrics.iloc[0]['Customer ID']
print(f"\n--- Deep Dive into Customer {sample_return_customer} ---")
print(f"Aggregated Returns: \n{returns_metrics[returns_metrics['Customer ID'] == sample_return_customer]}")
print("Raw Return Transactions:")
print(returns_raw[returns_raw['Customer ID'] == sample_return_customer][['Invoice', 'Quantity', 'TotalAmount']])